# **EQUIPO LFM** | Aprendizaje automatico

## **Introducción**

El presente trabajo tiene como objetivo desarrollar una metodología analítica que permita predecir el ancho del pétalo de la especie botánica Iris utilizando KNN y regresión lineal sobre el dataset Iris que contiene datos de tres subespecies.

Se evalúan los resultados de los entrenamientos realizados en una, dos y tres dimensiones, explorando las diferentes combinaciones que el dataset de origen permite. Esto busca obtener un mayor entendimiento de cómo la selección dimensional y la correlación entre variables afectan la precisión de los modelos obtenidos.

Es importante mencionar que, con fines pedagógicos que incluyen la exploración metodológica y la transmisión horizontal de conocimiento entre los miembros del equipo, este estudio excede los requerimientos mínimos planteados por la cátedra sin alejarse del eje temático central.

## Metodología

**El hiperparámetro K**
El presenta estudio considera tres tipos de hiperparametro K. 
- "K teórico" que deriva de la formula de cantidad de registros^(1/2).
- "K mínimo teórico" que es el que devuelve el mínimo MSE (error cuadratico medio) en la curva de exploración K/MSE.
- "K óptimo": el K con residuo menor que pertenece a la zona de estabilidad detectada analíticamente en la curva K/MSE.

**Criterio de Selección de K Basado en estabilidad local:**
En la optimización del modelo KNN, la práctica convencional dicta la selección del hiperparámetro k que minimiza la función de pérdida, en este caso, el Error Cuadrático Medio (MSE). No obstante, para garantizar la generalización del modelo y mitigar la sensibilidad ante el ruido estocástico de la muestra de entrenamiento, este estudio implementa un criterio de Zona de Estabilidad Analítica.
Por este motivo y en base a las evidencias del TP anterior que validan la teoría expuesta en la cátedra sobre el poco valor que tiene el uso del "K mínimo teorico", ya que frecuentemente reside en valles de alta sensibilidad, es que solo se lo identifica y se lo menciona en cada caso con fines documentales. Así, este trabajo emplea en su lugar el "K óptimo" para entrenar los modelos y su R^2 luego se contrasta con el R^2 obtenido empleando el K teórico.
Para detectar la zona de estabilidad local se emplea una función que identifica ventanas no menores a tres K contiguos para los que la diferencia MSE no supere un umbral que se establece con el valor base 0.00125 que deriva de realizar una partición diádica de la unidad de resolución mínima del fenómeno MSE observado, es decir 0.001/8 = 0.00125. Si la función no encuentra un rango de valores de K con ese umbral, realiza una nueva exploración duplicando el mismo sucesivamente hasta encontrar una zona de estabilidad dentro del umbral.
Una vez detectado el rango de K, la función selecciona el que esta asociado al menor MSE como "K óptimo".

**Estudio de las correlaciones**
Durante el EDA se confecciona una matriz de correlación de las variables del dataset Iris que es la clave fundamental para comprender el impacto de la selección de las variables para los diferentes experimentos llevados a cabo en el trabajo práctico y las conclusiones derivadas.



## Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import ShuffleSplit, cross_val_score
from mpl_toolkits.mplot3d import Axes3D


## Funciones

In [ ]:
# @title
# funciones

# Optimización de K: Error vs. Número de Vecinos
def koptimomse(X_train, X_test, y_train, y_test):
  k_values = range(1, 21) # Probamos k desde 1 hasta 20
  errores_mse = []

  for k in k_values:
      # Entrenar modelo con el k actual
      modelo = KNeighborsRegressor(n_neighbors=k)
      modelo.fit(X_train, y_train)

      # Predecir y calcular el error
      predicciones = modelo.predict(X_test)
      mse = mean_squared_error(y_test, predicciones)
      errores_mse.append(mse)

  # Guardar resultados y graficar
  resultados = pd.DataFrame({'k': k_values, 'MSE': errores_mse})

  plt.figure(figsize=(10, 6))
  plt.plot(k_values, errores_mse, marker='o', linestyle='--', color='darkblue')

  # Estética del gráfico
  plt.title('Optimización de K: Error vs. Número de Vecinos', fontsize=14)
  plt.xlabel('Valor de K (n_neighbors)', fontsize=12)
  plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
  plt.xticks(k_values) # Para ver todos los números del 1 al 20
  plt.grid(True, alpha=0.3)

  # Marcar el punto óptimo
  min_mse = min(errores_mse)
  best_k = k_values[errores_mse.index(min_mse)]
  plt.annotate(f'K para MSE mínimo teórico= {best_k}',
              xy=(best_k, min_mse),
              xytext=(best_k + 1, min_mse + 0.01),
              arrowprops=dict(facecolor='black', shrink=0.05))

  plt.show()
  print(f"El valor de K teórico con MSE mínimo es: {best_k}")
  return resultados, best_k


# Búsqueda del K con menor promedio de residuos
def koptimo_residuos(zona_estable, X_train, X_test, y_train, y_test):

  residuos_resultados = {}

  for i in range(len(zona_estable)):
      best_k = zona_estable[i]
      knn = KNeighborsRegressor(n_neighbors= best_k)
      knn.fit(X_train, y_train)
      y_pred = knn.predict(X_test)
      residuos = y_test - y_pred
      residuos_mean = residuos.mean()
      print(f"K: {best_k} | Prom residuos: {residuos_mean}")
      # print(f"Prom residuos: {residuos_mean}")
      residuos_resultados[best_k] = residuos_mean
  print("\n----------------\n")
  # Encontrar la clave asociada al valor mínimo
  best_k = min(residuos_resultados, key=residuos_resultados.get)

  # Obtener el valor mínimo usando esa clave
  min_residuo = residuos_resultados[best_k]

  print(f"Clave (K) con residuo mínimo: {best_k}")
  print(f"Valor del residuo mínimo: {min_residuo}")

  return best_k, min_residuo



# Cálculo de métricas de regresion
def regre(y_test, y_pred):
  # 1. Cálculo de métricas
  mse = mean_squared_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)
  return mse, r2


# Grafico de regresion con resultados
def regrapho(y_test, y_pred):
  # llamado a funcion 'regre' para obtener metricas
  mse, r2 = regre(y_test, y_pred)

  # 2. Generación del gráfico
  plt.figure(figsize=(8, 6))
  plt.scatter(y_test, y_pred, alpha=0.7, color='teal')
  plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', lw=2)

  # Configuración de etiquetas
  plt.xlabel('Valores Reales')
  plt.ylabel('Predicciones')
  plt.title('Real vs Predicho (KNN Regressor)')

  # 3. Agregar el cuadro de texto "dentro" del gráfico (esquina inferior derecha)
  texto_metricas = f"MSE: {mse:.4f}\nR²: {r2:.4f}"

  # Usamos plt.gca().text para posicionar el texto
  # transform=plt.gca().transAxes asegura que (1,0) sea la esquina inferior derecha exacta
  plt.gca().text(0.95, 0.05, texto_metricas,
                transform=plt.gca().transAxes,
                fontsize=11,
                verticalalignment='bottom',
                horizontalalignment='right',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='teal'))

  plt.show()
  print(f"Mean Square Error: {mse}")
  print(f"R^2: {r2}")
  return mse, r2


# Función para Línea de Regresión 2D (Experimentos con 1 variable)
def graficar_linea_2d_knn(modelo, X, y, nombres_cols):
    """
    Genera un gráfico 2D con los puntos reales y la línea de predicción escalonada de KNN.
    X: DataFrame o Array con exactamente 1 columna.
    """
    plt.figure(figsize=(10, 6))

    # Datos reales
    x_data = X.iloc[:, 0] if hasattr(X, 'iloc') else X.flatten()
    plt.scatter(x_data, y, color='gray', alpha=0.5, label='Datos Reales')

    # Crear línea de predicción suave
    x_range = np.linspace(x_data.min(), x_data.max(), 500).reshape(-1, 1)

    # Manejo de nombres de columnas si es necesario
    if hasattr(X, 'columns'):
        import pandas as pd
        x_range_df = pd.DataFrame(x_range, columns=X.columns)
        y_pred = modelo.predict(x_range_df)
    else:
        y_pred = modelo.predict(x_range)

    plt.plot(x_range, y_pred, color='red', linewidth=2, label='Línea de Regresión KNN')

    plt.xlabel(nombres_cols[0])
    plt.ylabel(nombres_cols[1])
    plt.title('Análisis de Regresión KNN (2D)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


# Función para Superficie 3D (Experimentos con 2 variables)
def graficar_superficie_3d_knn(modelo, X, y, nombres_cols):
    """
    Genera un gráfico 3D con los puntos reales y la superficie de predicción de KNN.
    X: DataFrame o Array con exactamente 2 columnas.
    y: Series o Array con la variable objetivo.
    nombres_cols: Lista con [Nombre X1, Nombre X2, Nombre Y].
    """
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # 1. Datos reales
    # Si X es DataFrame, usamos .iloc para evitar errores de índice
    x1_data = X.iloc[:, 0] if hasattr(X, 'iloc') else X[:, 0]
    x2_data = X.iloc[:, 1] if hasattr(X, 'iloc') else X[:, 1]
    ax.scatter(x1_data, x2_data, y, c='teal', marker='o', alpha=0.6, label='Datos Reales')

    # 2. Crear malla para la superficie de predicción
    x1_range = np.linspace(x1_data.min(), x1_data.max(), 30)
    x2_range = np.linspace(x2_data.min(), x2_data.max(), 30)
    X1_mesh, X2_mesh = np.meshgrid(x1_range, x2_range)

    # Crear set de prueba para la malla
    malla_input = np.c_[X1_mesh.ravel(), X2_mesh.ravel()]

    # Predicción sobre la malla
    # Nota: Si el modelo se entrenó con nombres de columnas, convertimos a DF
    if hasattr(X, 'columns'):
        import pandas as pd
        malla_df = pd.DataFrame(malla_input, columns=X.columns)
        Z_mesh = modelo.predict(malla_df).reshape(X1_mesh.shape)
    else:
        Z_mesh = modelo.predict(malla_input).reshape(X1_mesh.shape)

    # 3. Graficar superficie
    surf = ax.plot_surface(X1_mesh, X2_mesh, Z_mesh, cmap='viridis', alpha=0.4, edgecolor='none')

    ax.set_xlabel(nombres_cols[0])
    ax.set_ylabel(nombres_cols[1])
    ax.set_zlabel(nombres_cols[2])
    plt.title(f'Superficie de Regresión KNN (3D)')
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
    plt.show()


def graficar_histograma_residuos(y_real, y_pred):
    """
    Grafica la distribución de los errores (residuos).
    y_real: Valores verdaderos del set de test.
    y_pred: Predicciones hechas por el modelo KNN.
    """
    residuos = y_real - y_pred

    plt.figure(figsize=(10, 6))
    sns.histplot(residuos, kde=True, color='salmon', bins=20)

    # Línea de referencia en el cero
    plt.axvline(x=0, color='black', linestyle='--', label='Error Cero')

    plt.title('Distribución de Residuos (Errores del Modelo KNN)', fontsize=14)
    plt.xlabel('Valor del Error (Real - Predicho)', fontsize=12)
    plt.ylabel('Frecuencia', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # Diagnóstico rápido
    print(f"Sesgo promedio (Media de residuos): {residuos.mean():.4f}")


def zona_estabilidad_continua(df, umbral_p2p=0.005, min_elementos=3):
    """
    df: DataFrame con columnas 'k' y 'MSE'
    umbral_p2p: Máxima diferencia permitida entre un K y el siguiente
    min_elementos: Cantidad mínima de Ks consecutivos para considerar una zona estable
    """
    df = df.sort_values('k').copy()
    intentos = 0
    max_intentos = 5 # Guardrail para evitar bucles infinitos si los datos son erráticos
    
    while intentos < max_intentos:
        # 1. Calculamos la diferencia absoluta punto a punto
        df['diff_next'] = df['MSE'].diff().abs().shift(-1)

        # 2. Creamos una máscara de puntos estables
        df['es_estable'] = df['diff_next'] < umbral_p2p

        # 3. Identificar bloques continuos
        df['group_id'] = (df['es_estable'] != df['es_estable'].shift()).cumsum()

        # 4. Filtrar grupos estables
        estables = df[df['es_estable'] == True]

        mejor_zona = []
        max_len = 0

        if not estables.empty:
            for g_id, group in estables.groupby('group_id'):
                ks_del_grupo = group['k'].tolist()
                
                # Incluimos el último elemento del enlace (si K=12 es estable, implica que K=13 también lo es)
                ultimo_k = ks_del_grupo[-1] + 1
                if ultimo_k in df['k'].values:
                    ks_del_grupo.append(ultimo_k)

                # Guardamos el bloque más largo encontrado con este umbral
                if len(ks_del_grupo) > max_len:
                    max_len = len(ks_del_grupo)
                    mejor_zona = ks_del_grupo

        # --- NUEVA LÓGICA DE CONDICIÓN ---
        
        # Si encontramos una zona y cumple con el mínimo de elementos, la retornamos
        if len(mejor_zona) >= min_elementos:
            print(f"Zona encontrada con umbral {umbral_p2p}: {mejor_zona}")
            return mejor_zona
        
        # Si no se cumple, duplicamos el umbral y volvemos a intentar
        print(f"Umbral {umbral_p2p} insuficiente (zona de tamaño {len(mejor_zona)}). Reintentando...")
        umbral_p2p *= 2
        intentos += 1

    return mejor_zona # Retorna lo mejor que haya encontrado tras los intentos


## Data acquisition

In [ ]:
dfIris = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vSLgU6YF5djPgcJvcmXyqdIjfVefPsYlj6HUnRH15sZwsEL4GX7KPY-c3CWgM3n8vCljid-ZPocdAAl/pub?output=csv')

## EDA - Preprocesamiento

In [ ]:
dfIris

### Feature Scaling
- No se hace uso de un StandardScaler o MinMaxScaler para unificar las escalas porque en el dataset estas ya estan informadas en la misma escala (centímetros).

### Header standarization & data cleaning

- Renombraremos las columnas al español, y eliminaremos la columan Id ya que no será utilizada para el análisis

In [ ]:
#Renombrado de columnas
dfIris.rename({'SepalLengthCm':'sepalo_largo',
           'SepalWidthCm':'sepalo_ancho',
           'PetalLengthCm':'petalo_largo',
           'PetalWidthCm':'petalo_ancho',
           'Species':'especies'},
          axis=1, inplace=True) # inplace = True para que el renombrado sea sobre el mismo df

dfIris.drop('Id', axis=1, inplace=True) #axis=1 indica que es una columna


### Inspección del df procesado

In [ ]:
dfIris.shape

In [ ]:
dfIris

## EDA - Matriz de Correlación

In [ ]:
# 1. Calculamos la correlación de las columnas numéricas
matriz_corr = dfIris.drop(columns=['especies']).corr()

# 2. Generamos el gráfico
plt.figure(figsize=(8, 6))
sns.heatmap(matriz_corr, annot=True, cmap='RdYlGn', center=0, square=True)

plt.title('Matriz de Correlación: ¿Quién influye más en el ancho del pétalo?')
plt.show()

## Observaciones parciales 1:

- Largo del pétalo: Según se observa en la matriz de correlación es la variable con mayor correlación con el ancho del pétalo.
- Largo del sépalo: Esta variable también tiene una alta correlación con el ancho del petalo.
- Ancho del sépalo: Esta variable tiene una correlación muy baja e incluso negativa con el ancho del pétalo.

## Estudio de impacto de la dimensionalidad en los entrenamientos

Se procede a entrenar modelos con cantidades diferentes sucesivas de dimensiones y evaluar su rendimiento.



## Tres dimensiones

- Largo del pétalo
- Ancho del sépalo
- Largo del sépalo



### Particionamiento

In [ ]:
#X3 = dfIris.drop(['petalo_ancho', 'especies'], axis=1)
X3 = dfIris.drop(['petalo_ancho'], axis=1)
y = dfIris['petalo_ancho']

In [ ]:
# 2. División en entrenamiento y prueba
X_train3, X_test3, y_train3, y_test = train_test_split(X3, y, test_size=0.2, random_state=42)

In [ ]:
print('Población relativa de las diferentes especies en la muestra obtenida:')
# Muestra el conteo absoluto de cada especie
print(X_test3['especies'].value_counts())

# O de forma más elegante para ver porcentajes (representatividad relativa)
print(X_test3['especies'].value_counts(normalize=True) * 100)
X_test3.drop('especies', axis=1, inplace=True)
X_train3.drop('especies', axis=1, inplace=True)

### Búsqueda de K óptimo - Holdout - K vs MSE

In [ ]:
resultados_k3, best_k3_teorico = koptimomse(X_train3, X_test3, y_train3, y_test)

In [ ]:
# intento nueva detección de zona estable
mse_test, r2_test = regre(y_test, y_pred3)
zona_estable_k3_test = zona_estabilidad_continua2(resultados_k3, tolerancia_relativa=0.01, min_elementos=3)
zona_estable_k3_test


In [ ]:
# --- Ejecución ---
# Ajusta el umbral_p2p según la escala de tu MSE (ej. 0.0004 o 0.0005)
zona_estable_k3 = zona_estabilidad_continua(resultados_k3, umbral_p2p=0.00125)
print(f"Rango de K en zona de estabilidad continua: {zona_estable_k3}")

print("\n--------------------\n")
print("Promedio de residuos para cada K de la zona de estabilidad\n")

best_k3, min_residuo_3 = koptimo_residuos(zona_estable_k3, X_train3, X_test3, y_train3, y_test)

### Implementación de KNN Regresor

In [ ]:
knn_3 = KNeighborsRegressor(n_neighbors=best_k3)
knn_3.fit(X_train3, y_train3)

### Predicciones

In [ ]:
y_pred3 = knn_3.predict(X_test3)

### Evaluación del modelo - Resultados (Graficando Línea de Identidad)

In [ ]:
mse3, r2_3 = regrapho(y_test, y_pred3)

### Histograma de Residuos

In [ ]:
graficar_histograma_residuos(y_test, y_pred3)

## Dos dimensiones


### Particionamiento A

Para el primer experimento con dos dimensiones tomamos una variable con fuerte correlación y otra de correlación débil con respecto a nuetra variable objetivo

- Largo del pétalo
- Ancho del sépalo

In [ ]:
X2a = dfIris.drop(['petalo_ancho', 'especies', 'sepalo_largo'], axis=1)
nombres_2a = ['Largo del pétalo', 'Ancho del sépalo', 'Petalo ancho']

In [ ]:
# 2. División en entrenamiento y prueba
X_train2a, X_test2a, y_train2a, y_test = train_test_split(X2a, y, test_size=0.2, random_state=42)

#### Búsqueda de K óptimo - Holdout - K vs MSE

In [ ]:
resultados_k2a, best_k2a_teorico = koptimomse(X_train2a, X_test2a, y_train2a, y_test)

In [ ]:
# --- Ejecución ---
# Ajusta el umbral_p2p según la escala de tu MSE (ej. 0.0004 o 0.0005)
zona_estable_k2a = zona_estabilidad_continua(resultados_k2a, umbral_p2p=0.00125)
print(f"Rango de K en zona de estabilidad continua: {zona_estable_k2a}")

print("\n--------------------\n")
print("Promedio de residuos para cada K de la zona de estabilidad\n")

best_k2a, min_residuo_2a = koptimo_residuos(zona_estable_k2a, X_train2a, X_test2a, y_train2a, y_test)

#### Implementación KNN Regressor

In [ ]:
knn_2a = KNeighborsRegressor(n_neighbors=best_k2a)
knn_2a.fit(X_train2a, y_train2a)

#### Predicciones

In [ ]:
y_pred2a = knn_2a.predict(X_test2a)

#### Evaluación del modelo - Resultados (Graficando Línea de Identidad)



In [ ]:
mse2a, r2_2a = regrapho(y_test, y_pred2a)

#### Gráfico de Línea de Regresión

In [ ]:
graficar_superficie_3d_knn(knn_2a, X_test2a, y_test, nombres_2a)

#### Histograma de Residuos

In [ ]:
graficar_histograma_residuos(y_test, y_pred2a)

### Particionamiento b

En el segundo experimento de dos dimensiones tomamos las dos variables que tienen fuerte correlación con nuestra variable objetivo

- Largo del pétalo
- Largo del sépalo

In [ ]:
X2b = dfIris.drop(['petalo_ancho', 'especies', 'sepalo_ancho'], axis=1)
nombres_2b = ['Largo del pétalo', 'Largo del sépalo', 'Petalo ancho']

In [ ]:
# 2. División en entrenamiento y prueba
X_train2b, X_test2b, y_train2b, y_test = train_test_split(X2b, y, test_size=0.2, random_state=42)

#### Búsqueda de K óptimo - Holdout - K vs MSE

In [ ]:
resultados_k2b, best_k2b = koptimomse(X_train2b, X_test2b, y_train2b, y_test)

In [ ]:
# --- Ejecución ---
# Ajusta el umbral_p2p según la escala de tu MSE (ej. 0.0004 o 0.0005)
zona_estable_k2b = zona_estabilidad_continua(resultados_k2b, umbral_p2p=0.00125)
print(f"Rango de K en zona de estabilidad continua: {zona_estable_k2b}")

print("\n--------------------\n")
print("Promedio de residuos para cada K de la zona de estabilidad\n")

best_k2b, min_residuo_2b = koptimo_residuos(zona_estable_k2b, X_train2b, X_test2b, y_train2b, y_test)

#### Implementación KNN Regressor

In [ ]:
knn_2b = KNeighborsRegressor(n_neighbors=best_k2b)
knn_2b.fit(X_train2b, y_train2b)

#### Predicciones

In [ ]:
y_pred2b = knn_2b.predict(X_test2b)

#### Evaluación del modelo - Resultados

In [ ]:
mse2b, r2_2b = regrapho(y_test, y_pred2b)


#### Gráfico de Línea de Regresión

In [ ]:
graficar_superficie_3d_knn(knn_2b, X_test2b, y_test, nombres_2b)

#### Histograma de Residuos

In [ ]:
graficar_histograma_residuos(y_test, y_pred2b)

La diferencia en los valores de $k$ (10 vs 14) se debe a cómo cambian las "vecindades" en el espacio multidimensional cuando cambiamos las variables.

 Experimento 1: Fuerte (+) y Negativa (-) ($k=10$)Al combinar una variable que "empuja" en la misma dirección que el objetivo (largo del pétalo) con una que "tira" en contra (ancho del sépalo), estamos creando un espacio con mucha estructura y contraste. El efecto: Las especies de flores quedan muy separadas entre sí. Como la separación es clara y hay "contraste" gracias a la variable negativa, el modelo encuentra el punto de equilibrio rápido. Por qué $k=10$: Necesitamos menos vecinos para promediar porque la estructura es robusta; hay menos ambigüedad en quién es vecino de quién.

 Experimento 2: Dos Fuertes (+) ($k=14$)Al usar dos variables con fuerte correlación (ej. largo del pétalo y largo del sépalo), tenemos redundancia. Ambas variables dicen casi lo mismo. El efecto: Esto crea lo que se llama "densidad" o "ruido de redundancia". Los puntos están muy pegados y las fronteras entre grupos de flores se vuelven más borrosas porque ambas variables escalan igual. Por qué $k=14$: Al haber más "amontonamiento" de datos que dicen lo mismo, el modelo necesita un $k$ más alto para suavizar la predicción. Un $k$ mayor ayuda a ignorar las pequeñas fluctuaciones de esos datos redundantes y a obtener un promedio más estable.

 En resumen:

 - Con contraste (+ y -): El modelo es más "decidido" y necesita mirar a menos vecinos para acertar.

 - Con redundancia (+ y +): El modelo se vuelve más "indeciso" porque los datos se parecen demasiado entre sí, y necesita ampliar el círculo de búsqueda ($k$ más alto) para estar seguro de la predicción.

### Particionamiento C

En el tercer experimento de dos dimensiones tomamos las variables

- Largo del sépalo
- Ancho del sépalo

La primera tiene una correlación fuerte con ancho del pétalo y ancho del sépalo que tiene una correlación débil y negativa.

In [ ]:
X2c = dfIris.drop(['petalo_largo', 'especies', 'petalo_ancho'], axis=1)
nombres_2c = ['Largo del sépalo', 'Ancho del sépalo', 'Petalo ancho']

In [ ]:
# 2. División en entrenamiento y prueba
X_train2c, X_test2c, y_train2c, y_test = train_test_split(X2c, y, test_size=0.2, random_state=42)

#### Búsqueda de K óptimo - Holdout - K vs MSE

In [ ]:
resultados_k2c, best_k2c = koptimomse(X_train2c, X_test2c, y_train2c, y_test)

In [ ]:
# --- Ejecución ---
# Ajusta el umbral_p2p según la escala de tu MSE (ej. 0.0004 o 0.0005)
zona_estable_k2c = zona_estabilidad_continua(resultados_k2c, umbral_p2p=0.00125)
print(f"Rango de K en zona de estabilidad continua: {zona_estable_k2c}")

print("\n--------------------\n")
print("Promedio de residuos para cada K de la zona de estabilidad\n")

best_k2c, min_residuo_2c = koptimo_residuos(zona_estable_k2c, X_train2c, X_test2c, y_train2c, y_test)

#### Implementación KNN Regressor

In [ ]:
KNN_2c = KNeighborsRegressor(n_neighbors=best_k2c)
KNN_2c.fit(X_train2c, y_train2c)

#### Predicciones

In [ ]:
y_pred2c = KNN_2c.predict(X_test2c)

#### Evaluación modelo - Resultados

In [ ]:
mse2c, r2_2c = regrapho(y_test, y_pred2c)

#### Gráfico de Linea de Regresión

In [ ]:
graficar_superficie_3d_knn(KNN_2c, X_test2c, y_test, nombres_2c)

#### Histograma de Residuos

In [ ]:
graficar_histograma_residuos(y_test, y_pred2c)

## Una dimension


### Particionamiento a

En este primer experimento de una sola dimensión comenzamos primero con la variable de correlación más fuerte

- Largo del pétalo

In [ ]:
X1a = dfIris.drop(['petalo_ancho', 'especies', 'sepalo_largo', 'sepalo_ancho'], axis=1)
nombres_1a = ['Largo del pétalo', 'Petalo ancho']

In [ ]:
# 2. División en entrenamiento y prueba
X_train1a, X_test1a, y_train1a, y_test = train_test_split(X1a, y, test_size=0.2, random_state=42)

#### Búsqueda de K óptimo - Holdout - K vs MSE

In [ ]:
resultados_1a, best_k1a = koptimomse(X_train1a, X_test1a, y_train1a, y_test)

In [ ]:
# --- Ejecución ---
# Ajusta el umbral_p2p según la escala de tu MSE (ej. 0.0004 o 0.0005)
zona_estable_1a = zona_estabilidad_continua(resultados_1a, umbral_p2p=0.00125)
print(f"Rango de K en zona de estabilidad continua: {zona_estable_1a}")

print("\n--------------------\n")
print("Promedio de residuos para cada K de la zona de estabilidad\n")

best_k1a, min_residuo_1a = koptimo_residuos(zona_estable_1a, X_train1a, X_test1a, y_train1a, y_test)

#### Implementación de KNN Regressor

In [ ]:
knn_1a = KNeighborsRegressor(n_neighbors=best_k1a)
knn_1a.fit(X_train1a, y_train1a)

#### Predicciones

In [ ]:
y_pred1a = knn_1a.predict(X_test1a)

#### Evaluación del modelo - Resultados


In [ ]:
mse1a, r2_1a = regrapho(y_test, y_pred1a)

#### Gráfico de Línea de Regresión

In [ ]:
graficar_linea_2d_knn(knn_1a, X_test1a, y_test, nombres_1a)

#### Histograma de Residuos

In [ ]:
graficar_histograma_residuos(y_test, y_pred1a)

### Particionamiento b
En el segundo experimento de una sola dimensión tomamos la segunda variable de correlación más fuerte

- Largo del sépalo

In [ ]:
X1b = dfIris.drop(['petalo_ancho', 'especies', 'petalo_largo', 'sepalo_ancho'], axis=1)
nombres_1b = ['Largo del sépalo', 'Sepalo largo']

In [ ]:
# 2. División en entrenamiento y prueba
X_train1b, X_test1b, y_train1b, y_test = train_test_split(X1b, y, test_size=0.2, random_state=42)

#### Búsqueda de K óptimo - Holdout - K vs MSE

In [ ]:
resultados_1b, best_k1b = koptimomse(X_train1b, X_test1b, y_train1b, y_test)

In [ ]:
# --- Ejecución ---
# Ajusta el umbral_p2p según la escala de tu MSE (ej. 0.0004 o 0.0005)
zona_estable_1b = zona_estabilidad_continua(resultados_1b, umbral_p2p=0.00250)
print(f"Rango de K en zona de estabilidad continua: {zona_estable_1b}")

print("\n--------------------\n")
print("Promedio de residuos para cada K de la zona de estabilidad\n")

best_k1b, min_residuo_1b = koptimo_residuos(zona_estable_1b, X_train1b, X_test1b, y_train1b, y_test)

#### Implementación de KNN Regressor

In [ ]:
knn_1b = KNeighborsRegressor(n_neighbors=best_k1b)
knn_1b.fit(X_train1b, y_train1b)

#### Predicciones

In [ ]:
y_pred1b = knn_1b.predict(X_test1b)

#### Evaluación del modelo - Resultados


In [ ]:
mse1b, r2_1b = regrapho(y_test, y_pred1b)

#### Gráfico de Línea de Regresión

In [ ]:
graficar_linea_2d_knn(knn_1b, X_test1b, y_test, nombres_1b)

#### Histograma de Residuos

In [ ]:
graficar_histograma_residuos(y_test, y_pred1b)

### Particionamiento c
En el tercer experimento de una sola dimensión tomamos la variable de correlación débil

- Ancho del sépalo

In [ ]:
X1c = dfIris.drop(['petalo_ancho', 'especies', 'petalo_largo', 'sepalo_largo'], axis=1)
nombres_1c = ['Ancho del sépalo', 'Sepalo ancho']

In [ ]:
# 2. División en entrenamiento y prueba
X_train1c, X_test1c, y_train1c, y_test = train_test_split(X1c, y, test_size=0.2, random_state=42)

#### Búsqueda de K óptimo - Holdout - K vs MSE

In [ ]:
resultados_1c, best_k1c = koptimomse(X_train1c, X_test1c, y_train1c, y_test)

In [ ]:
# --- Ejecución ---
# Ajusta el umbral_p2p según la escala de tu MSE (ej. 0.0004 o 0.0005)
zona_estable_1c = zona_estabilidad_continua(resultados_1c, umbral_p2p=0.00125)
print(f"Rango de K en zona de estabilidad continua: {zona_estable_1c}")

print("\n--------------------\n")
print("Promedio de residuos para cada K de la zona de estabilidad\n")

best_k1a, min_residuo_1a = koptimo_residuos(zona_estable_1c, X_train1c, X_test1c, y_train1c, y_test)

#### Implementación de KNN Regressor

In [ ]:
knn_1c = KNeighborsRegressor(n_neighbors=best_k1c)
knn_1c.fit(X_train1c, y_train1c)

#### Predicciones

In [ ]:
y_pred1c = knn_1c.predict(X_test1c)

#### Evaluación del modelo - Resultados

In [ ]:
mse1c, r2_1c = regrapho(y_test, y_pred1c)

#### Gráfico de Línea de Regresión

In [ ]:
graficar_linea_2d_knn(knn_1c, X_test1c, y_test, nombres_1c)

#### Histograma de Residuos

In [ ]:
graficar_histograma_residuos(y_test, y_pred1c)

# Resultados y comparaciones

A fin de observar el impacto de la exploración de K óptimos, se procede a obtener los valores de R^2 y MSE para todas las dimensionalidades usando el valor teórico de K acorde al tamaño del dataset (K = 12).

Con todos los datos se crea una tabla para poder realizar las comparaciones correspondientes.

In [ ]:
# Entrenamos los modelos con K teorico = 12
knn = KNeighborsRegressor(n_neighbors=12)

knn.fit(X_train3, y_train3)
y_pred_12n_3 = knn.predict(X_test3)
mse_12n_3, r2_12n_3 = regre(y_test, y_pred_12n_3)

knn.fit(X_train2a, y_train2a)
y_pred_12n_2a = knn.predict(X_test2a)
mse_12n_2a, r2_12n_2a = regre(y_test, y_pred_12n_2a)

knn.fit(X_train2b, y_train2b)
y_pred_12n_2b = knn.predict(X_test2b)
mse_12n_2b, r2_12n_2b = regre(y_test, y_pred_12n_2b)

knn.fit(X_train2c, y_train2c)
y_pred_12n_2c = knn.predict(X_test2c)
mse_12n_2c, r2_12n_2c = regre(y_test, y_pred_12n_2c)

knn.fit(X_train1a, y_train1a)
y_pred_12n_1a = knn.predict(X_test1a)
mse_12n_1a, r2_12n_1a = regre(y_test, y_pred_12n_1a)

knn.fit(X_train1b, y_train1b)
y_pred_12n_1b = knn.predict(X_test1b)
mse_12n_1b, r2_12n_1b = regre(y_test, y_pred_12n_1b)

knn.fit(X_train1c, y_train1c)
y_pred_12n_1c = knn.predict(X_test1c)
mse_12n_1c, r2_12n_1c = regre(y_test, y_pred_12n_1c)

# Creamos un diccionario con los resultados

data = {
    "Dimensiones": ["3 dimensiones", "2 dimensiones-A", "2 dimensiones-B",  "2 dimensiones-C", "1 dimensión-A", "1 dimensión-B", "1 dimensión-C"],
    "| ": ['| '] * 7,
    "K optimo": [best_k3, best_k2a, best_k2b, best_k2c, best_k1a, best_k1b, best_k1c],
    " ": [' '] * 7,
    "MSE": [mse3, mse2a, mse2b, mse2c, mse1a, mse1b, mse1c],
    "R² Score": [r2_3, r2_2a, r2_2b, r2_2c, r2_1a, r2_1b, r2_1c],
    " | ": ['| '] * 7,
    "K teórico": ['12 '] * 7,
    " MSE": [mse_12n_3, mse_12n_2a, mse_12n_2b, mse_12n_2c, mse_12n_1a, mse_12n_1b, mse_12n_1c],
    " R² Score": [r2_12n_3, r2_12n_2a, r2_12n_2b, r2_12n_2c, r2_12n_1a, r2_12n_1b, r2_12n_1c]
}

# Se redondean los valores de 'data' a 4 posiciones luego de la coma
data_formateada = {
    key: [round(val, 4) if isinstance(val, float) else val for val in values]
    for key, values in data.items()
}

# Creamos el DataFrame
df_comparativo = pd.DataFrame(data_formateada)

# Imprimimos con un formato estético
print("TABLA COMPARATIVA DE MODELOS")
print("-" * 80)
print(df_comparativo.to_string(index=False))
print("-" * 80)

# Opcional: Si quieres resaltar cuál es el mejor (menor MSE)
mejor_modelo = df_comparativo.loc[df_comparativo['MSE'].idxmin(), 'Dimensiones']
peor_modelo = df_comparativo.loc[df_comparativo['MSE'].idxmax(), 'Dimensiones']
print(f"El mejor desempeño general lo tiene el modelo: {mejor_modelo}")
print(f"El peor desempeño general lo tiene el modelo: {peor_modelo}")

# Imprimir el nombre del modelo que con K teórico es mejor que con K óptimo
mejor_modelo_teorico = df_comparativo.loc[df_comparativo[' MSE'].idxmin(), 'Dimensiones']


In [ ]:
# 1. Identificamos los casos donde el R² Score obtenido con K teórico es menor al R² Score óptimo
casos_mejor_teorico = df_comparativo[df_comparativo[' R² Score'] > df_comparativo['R² Score']]

print("Modelos en que R² Score con K teórico superó al K óptimo")
print("-" * 60)

if not casos_mejor_teorico.empty:
    for index, row in casos_mejor_teorico.iterrows():
        diff = row[' R² Score'] - row['R² Score']
        print(f"Modelo: {row['Dimensiones']}")
        print(f"  > R² Score con K Óptimo ({row['K optimo']}): {row['R² Score']:.4f}")
        print(f"  > R² Score con K Teórico (12): {row[' R² Score']:.4f}")
        print(f"  > Mejora del teórico: {diff:.4f}")
        print("-" * 30)
else:
    print("En ninguna combinación el K teórico fue más performante.")

# 2. Guardar los nombres en una lista si necesitas usarlos luego
nombres_modelos_mejor_teorico = casos_mejor_teorico['Dimensiones'].tolist()

# Conclusión:

**Sobre las correlaciones**

Las observaciones sobre las correlaciones entre las diferentes variables de este dataset sobre largo y ancho de sepalos y petalos mantienen coherencia biológica, ya que los petalos así como los sepalos, crecen en ambas direcciones, a lo largo y a lo ancho.
Los sépalos (cáliz, exterior) y pétalos (corola, interior) son hojas modificadas que forman el perianto que tiene una función de protección de los organos sexuales y de atracción de polinizadores.
Las correlación Sépalo Largo vs. Pétalo Largo (0.87) y Pétalo Ancho (0.82) dan cuenta del desarrollo conjunto del perianto en la especie Iris ya que si el sépalo (que protege al capullo) es grande, biológicamente es esperable que el pétalo que contiene también lo sea.
Resulta destacable, sin embargo, que el ancho del sépalo tiene correlaciones negativas o muy débiles con el resto (-0.11, -0.42, -0.36). Esto sugiere que, en la familia Iris, el ancho del sépalo sigue una lógica que en algún punto del desarrollo se vuelve distinta a las otras tres variables.

**En cuanto a la metodología empleada para la elección de K**

Los resultados indican que la selección analítica del K óptimo devuelve mejores resultados que superan o igualan los valores de R² obtenidos con el K teórico a nivel de la segunda y tercera cifra significativa. Sin embargo, existen dos excepciones. En la primera, la diferencia es en la tercer cifra significativa con lo cual resulta prácticamente despreciable y en el otro se trata de una diferencia de 3 milésimas que tiene impacto en la segunda cifra significativa. Estos hallazgos consideramos que resultan una validación del método analítico propuesto e invitan a una discusión que se comenta en la sección apropiada.
(Meto: euclidiana (Minkowski p=2 por defecto en Scikit learn)
**El problema de la multi-colinealidad en la multi-dimensinalidad:**
Una de las hipotesis fue que al entrenar el dataset con las tres dimensiones existiendo dos con alta colinealidad que son largo del pétalo y largo del sépalo, existía un peligro de multi-colinealidad que hiciera que la información se vuelva redundante y el cálculo de la distancia se vea "arrastrado" doblemente por la misma tendencia subyacente y el algoritmo resultara menos sensible a variables de menor correlación (como el ancho del sépalo) y la predicción termine dependiendo excesivamente de la longitud. Los resultados indicaron lo contrario y la regresión lineal obtenida con 3-dimensiones fue la mejor de todas, seguida por el modelo 2-dimensiones-A, 2-dimensiones-B y 1-dimensión-A, incluyendo todos ellos la variable pétalo-largo.
Por el contrario los modelos que no incluyeron la variable pétalo-largo, la de mayor correlación con pétalo-ancho, obtuvieron resultados menos performantes haciendo evidente lo adelantado por la matriz de correlación respecto a la fuerte correlación entre pétalo-largo y pétalo-ancho.

# Discusión:
**La excepción de correlación de la variable ancho del sépalo** 

Esto podría deberse a una correlación que esta mantiene con otra estructura de la anatomia vegetal no observada en este dataset o incluso a una variable temporal o de etapa del desarrollo que tampoco se encuentran observadas. Se hipotetiza que la función de soporte mecánico, impone restricciones físicas al crecimiento en ancho de los sépalos que las otras hojas modificadas (pétalos) no tienen y que posiblemente un determinado momento de la pre-antesis o el inicio de la antesis determinen un punto de inflexión en el comportamiento de la correlación de esta con las demás dimensiones observadas en el dataset Iris.

**Sobre la colinealidad y la búsqueda del K óptimo**

Aunque la colinealidad existe, los resultados indican que los modelos se beneficiaron de esta redundancia informativa para ganar estabilidad en la predicción al emplear KNN que mide distancia posiblemente actúando como un refuerzo de la señal frente al ruido.

Finalmente, la diferencia marginal entre el K óptimo analítico y el K teórico demuestra que, para este dataset, el costo computacional de la optimización no ofrece un retorno significativo. Esto se debe, probablemente, a la naturaleza altamente balanceada y la clara separabilidad de las clases en el dataset IrisLa diferencia de resultados obtenidos con el uso de un K óptimo respecto de uno teórico es indicativo de que el esfuerzo involucrado para diferenciar uno de otro no resulta relevante para este dataset. Posiblemente sea causa de ambas cuestiones la naturaleza tan balanceada del dataset Iris.
